# Official SnapUGC EVQA on 5000 Videos

This notebook runs the official SnapUGC implementation from `dasongli1/SnapUGC_Engagement` on the same 5000-video subset. This is the exact-paper check: EfficientNetV2 semantic features, UVQ-style distortion, ResNet3D action, mPLUG-2 caption/mid-layer features, YAMNet sound text, Stable Diffusion text encoder, and official `EVQA.pth`.

Run this on Kaggle GPU. The final metrics are written to `official_evqa_report.json`.

In [ ]:
# 0. Configuration
import os, sys, json, csv, glob, shutil, subprocess, textwrap, time
from pathlib import Path

MAX_VIDEOS = 5000
SUBSET_SEED = 42
ECR_BINS = 10
RESET_SUBSET = False  # set True if you want to rebuild subset_videos from scratch
COPY_VIDEOS = False   # symlink by default; set True only if symlink fails
PATCH_LIGHT_SD_TEXT_ENCODER = True  # same SD tokenizer/text_encoder, but avoids loading UNet/VAE
ALLOW_GOOGLE_DRIVE_CHECKPOINT_DOWNLOAD = False  # set True only if Drive quota is available

# The notebook needs TWO Kaggle inputs:
# 1) the SnapUGC dataset with train_data.csv and train_videos/
# 2) the official checkpoint dataset with EVQA.pth, mPLUG2_MSRVTT_Caption.pth,
#    net_distort6_g_latest.pth, r3d18_K_200ep.pth, and ViT-L-14.tar

def find_snapugc_input_root():
    preferred = [
        Path('/kaggle/input/datasets/nguyntuncng/snapugc-dataset'),
        Path('/kaggle/input/snapugc-dataset'),
    ]
    for root in preferred:
        if (root / 'train_data.csv').exists():
            return root
    candidates = [p.parent for p in Path('/kaggle/input').rglob('train_data.csv')]
    for root in candidates:
        if (root / 'train_videos').exists() or list(root.rglob('train_videos')):
            return root
    raise FileNotFoundError(
        'Cannot find SnapUGC dataset input. Attach the dataset that contains train_data.csv and train_videos/. '
        'Checkpoint-only input is not enough.'
    )

def find_train_video_root(input_root):
    candidates = [
        input_root / 'train_videos' / 'train_videos',
        input_root / 'train_videos',
    ]
    candidates += [p for p in input_root.rglob('train_videos') if p.is_dir()]
    for root in candidates:
        if root.is_dir() and any(root.glob('*.mp4')):
            return root
    raise FileNotFoundError(f'Cannot find mp4 files under train_videos in {input_root}')

INPUT_DIR = find_snapugc_input_root()
TRAIN_CSV_ORIG = INPUT_DIR / 'train_data.csv'
TRAIN_VIDEO_ROOT = find_train_video_root(INPUT_DIR)

WORK_DIR = Path('/kaggle/working')
OUTPUT_DIR = WORK_DIR / f'official_snapugc_evqa_{MAX_VIDEOS}'
SUBSET_VIDEO_DIR = OUTPUT_DIR / 'subset_videos'
SUBSET_CSV = OUTPUT_DIR / f'train_subset_{MAX_VIDEOS}.csv'
OFFICIAL_REPO_DIR = WORK_DIR / 'SnapUGC_Engagement'
OFFICIAL_COMMIT = '4e0ce3154225cfdf1d036e5b8b1d3874615a04f7'  # official repo HEAD checked on 2026-05-03
OFFICIAL_ECR_DIR = OFFICIAL_REPO_DIR / 'ECR_inference'
RUN_DIR = OUTPUT_DIR / 'official_run'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version)
print('TRAIN_CSV_ORIG:', TRAIN_CSV_ORIG, TRAIN_CSV_ORIG.exists())
print('TRAIN_VIDEO_ROOT:', TRAIN_VIDEO_ROOT, TRAIN_VIDEO_ROOT.is_dir())
print('OUTPUT_DIR:', OUTPUT_DIR)

if not TRAIN_CSV_ORIG.exists():
    raise FileNotFoundError(TRAIN_CSV_ORIG)
if not TRAIN_VIDEO_ROOT.is_dir():
    raise FileNotFoundError(TRAIN_VIDEO_ROOT)

In [ ]:
# 1. Install official dependencies
# Kaggle currently ships very new Python/torch. Keep transformers on a recent 4.x release:
# old 4.25 builds tokenizers from source, while transformers 5 can break mPLUG imports.
!pip install -q gdown decord ruamel.yaml tensorflow_hub tf-keras imageio matplotlib ipython oss2 timm scipy opencv-python-headless einops torchmetrics ftfy "transformers>=4.44,<5"

import transformers, tokenizers
print('transformers:', transformers.__version__)
print('tokenizers:', tokenizers.__version__)
major = int(transformers.__version__.split('.')[0])
if major >= 5:
    raise RuntimeError('transformers must be <5 for official mPLUG code; restart and rerun the install cell.')

import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not enabled. Turn on GPU accelerator before running official EVQA.')


In [ ]:
# 2. Create or reuse the same ECR-balanced 5000-video subset
import numpy as np
import pandas as pd

VIDEO_EXTENSIONS = {'.mp4', '.mov', '.mkv', '.webm', '.avi'}

def find_first_column(df, candidates):
    lower_to_actual = {str(c).lower(): str(c) for c in df.columns}
    for candidate in candidates:
        found = lower_to_actual.get(candidate.lower())
        if found is not None:
            return found
    return None

def build_video_index(video_dir):
    index = {}
    for path in Path(video_dir).iterdir():
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS:
            index.setdefault(path.stem, path)
            index.setdefault(path.name, path)
    return index

def attach_video_paths(df, video_dir, id_col):
    index = build_video_index(video_dir)
    rows = []
    for _, row in df.iterrows():
        video_id = str(row[id_col])
        path = index.get(video_id) or index.get(video_id + '.mp4')
        if path is None:
            continue
        item = row.copy()
        item['_video_path'] = str(path)
        rows.append(item)
    if not rows:
        raise RuntimeError(f'No CSV rows matched videos in {video_dir}')
    return pd.DataFrame(rows)

def allocate_quotas(group_sizes, total):
    group_ids = sorted(group_sizes)
    base = total // len(group_ids)
    quotas = {gid: min(size, base) for gid, size in group_sizes.items()}
    remaining = total - sum(quotas.values())
    while remaining > 0:
        progressed = False
        candidates = sorted(group_ids, key=lambda gid: (group_sizes[gid] - quotas[gid], group_sizes[gid]), reverse=True)
        for gid in candidates:
            if quotas[gid] >= group_sizes[gid]:
                continue
            quotas[gid] += 1
            remaining -= 1
            progressed = True
            if remaining == 0:
                break
        if not progressed:
            break
    return quotas

def stratified_sample_by_ecr(df, ecr_col, max_rows, bins=10, seed=42):
    work = df.copy()
    work[ecr_col] = pd.to_numeric(work[ecr_col], errors='coerce')
    work = work.dropna(subset=[ecr_col]).reset_index(drop=True)
    if len(work) <= max_rows:
        return work.sample(frac=1, random_state=seed).reset_index(drop=True)
    n_bins = min(int(bins), int(work[ecr_col].nunique()), max_rows)
    work['_ecr_bin'] = pd.qcut(work[ecr_col], q=n_bins, labels=False, duplicates='drop')
    work = work.dropna(subset=['_ecr_bin']).copy()
    work['_ecr_bin'] = work['_ecr_bin'].astype(int)
    groups = {int(gid): group for gid, group in work.groupby('_ecr_bin', sort=True)}
    quotas = allocate_quotas({gid: len(group) for gid, group in groups.items()}, max_rows)
    parts = []
    for gid, group in groups.items():
        take = quotas.get(gid, 0)
        if take > 0:
            parts.append(group.sample(n=take, random_state=seed + gid))
    sampled = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    return sampled.drop(columns=['_ecr_bin'], errors='ignore')

def materialize_subset(df, out_video_dir, id_col, reset=False, copy=False):
    out_video_dir = Path(out_video_dir)
    if reset and out_video_dir.exists():
        for path in out_video_dir.iterdir():
            if path.is_dir() and not path.is_symlink():
                shutil.rmtree(path)
            else:
                path.unlink(missing_ok=True)
    out_video_dir.mkdir(parents=True, exist_ok=True)
    for _, row in df.iterrows():
        src = Path(str(row['_video_path']))
        dst = out_video_dir / f'{row[id_col]}{src.suffix.lower()}'
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        if copy:
            shutil.copy2(src, dst)
        else:
            try:
                os.symlink(src, dst)
            except OSError:
                shutil.copy2(src, dst)

existing_videos = list(SUBSET_VIDEO_DIR.glob('*.mp4'))
if SUBSET_CSV.exists() and len(existing_videos) >= MAX_VIDEOS and not RESET_SUBSET:
    print('Reusing subset:', SUBSET_CSV, 'videos:', len(existing_videos))
else:
    df = pd.read_csv(TRAIN_CSV_ORIG)
    id_col = find_first_column(df, ['Id', 'id', 'video_id', 'videoid'])
    ecr_col = find_first_column(df, ['ECR', 'engagement', 'label', 'target'])
    if id_col is None or ecr_col is None:
        raise ValueError(f'Cannot find id/ecr columns in {TRAIN_CSV_ORIG}')
    available = attach_video_paths(df, TRAIN_VIDEO_ROOT, id_col)
    sampled = stratified_sample_by_ecr(available, ecr_col, MAX_VIDEOS, ECR_BINS, SUBSET_SEED)
    materialize_subset(sampled, SUBSET_VIDEO_DIR, id_col, reset=True, copy=COPY_VIDEOS)
    sampled.drop(columns=['_video_path'], errors='ignore').to_csv(SUBSET_CSV, index=False)
    print('Created subset:', SUBSET_CSV)

subset_df = pd.read_csv(SUBSET_CSV)
print('subset rows:', len(subset_df))
print('subset videos:', len(list(SUBSET_VIDEO_DIR.glob('*.mp4'))))
print(subset_df.head())
if len(subset_df) != MAX_VIDEOS:
    raise RuntimeError(f'Expected {MAX_VIDEOS} rows, got {len(subset_df)}')

In [ ]:
# 3. Clone official SnapUGC repo and prepare official checkpoints
if not (OFFICIAL_ECR_DIR / 'test_SnapUGC_baseline.py').exists():
    !git clone https://github.com/dasongli1/SnapUGC_Engagement.git "$OFFICIAL_REPO_DIR"
else:
    print('Official repo exists:', OFFICIAL_REPO_DIR)
subprocess.run(['git', 'checkout', OFFICIAL_COMMIT], cwd=str(OFFICIAL_REPO_DIR), check=True)

CHECKPOINT_DIR = OFFICIAL_ECR_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
official_drive_required = ['EVQA.pth', 'net_distort6_g_latest.pth', 'r3d18_K_200ep.pth', 'mPLUG2_MSRVTT_Caption.pth', 'ViT-L-14.tar']
# EfficientNetV2-s semantic features use this external ImageNet pretrained file.
# The original OneDrive URL inside modules/efficientnet_v2.py now returns 404, so
# put this file in your Kaggle input dataset under checkpoints/ as well.
external_pretrained_required = ['efficientnet_v2_s_21k_ft1k-dbb43f38.pth']
required = official_drive_required + external_pretrained_required
checkpoint_file_ids = {
    'EVQA.pth': '10Ymo7-wQJf7BeA3m-W3Hsc7z5ovn1-id',
    'mPLUG2_MSRVTT_Caption.pth': '12c0rPUQzChiWqCq6kV3Cne3kUfCXg3xr',
    'net_distort6_g_latest.pth': '16imkq1yHm8KdifAIWdj-HJMmVCGXv70C',
    'r3d18_K_200ep.pth': '1lKHBAfsqgN0hmeUwuryy1EX6pMZS6f3x',
    'ViT-L-14.tar': '1QIA_U0gM3gZCDCaELVCzarDkQay9A4ji',
}

# Prefer a Kaggle Input dataset containing the official checkpoint files. This avoids
# Google Drive quota and makes reruns deterministic. Typical paths are like:
# /kaggle/input/snapugc-official-checkpoints/EVQA.pth
# /kaggle/input/snapugc-official-checkpoints/ViT-L-14.tar
# /kaggle/input/snapugc-official-checkpoints/ECR_inference/checkpoints/EVQA.pth

def copy_checkpoint_from_kaggle_input(name):
    dst = CHECKPOINT_DIR / name
    if dst.exists() and dst.stat().st_size > 0:
        return True
    candidates = [p for p in Path('/kaggle/input').rglob(name) if p.is_file()]
    if candidates:
        src = candidates[0]
        print(f'Using Kaggle input checkpoint: {src} -> {dst}')
        try:
            os.symlink(src, dst)
        except OSError:
            shutil.copy2(src, dst)
        return True
    return False

for name in required:
    copy_checkpoint_from_kaggle_input(name)

missing = [name for name in required if not (CHECKPOINT_DIR / name).exists()]
print('Missing checkpoints before Google Drive download:', missing)

# Fallback: download only the missing required files one by one. Do not use
# gdown --folder, because it also tries ViT-L-14.tar and may stop early on quota.
# Default is disabled because the official Google Drive is frequently quota-blocked.
drive_missing = [name for name in missing if name in checkpoint_file_ids]
if drive_missing and ALLOW_GOOGLE_DRIVE_CHECKPOINT_DOWNLOAD:
    for name in list(drive_missing):
        file_id = checkpoint_file_ids[name]
        dst = CHECKPOINT_DIR / name
        print(f'Downloading missing checkpoint {name} from Google Drive...')
        result = subprocess.run(
            [sys.executable, '-m', 'gdown', f'https://drive.google.com/uc?id={file_id}', '-O', str(dst)],
            text=True,
        )
        if result.returncode != 0 and dst.exists():
            dst.unlink(missing_ok=True)
        copy_checkpoint_from_kaggle_input(name)
elif drive_missing:
    print('Google Drive checkpoint download is disabled. Set ALLOW_GOOGLE_DRIVE_CHECKPOINT_DOWNLOAD=True only if quota is available.')

missing = [name for name in required if not (CHECKPOINT_DIR / name).exists()]
print('Missing checkpoints after prepare:', missing)
if missing:
    raise FileNotFoundError(
        'Attach a Kaggle Dataset containing the required official checkpoint files, then rerun. Missing: '
        + ', '.join(missing)
        + '\nExpected filenames anywhere under /kaggle/input: '
        + ', '.join(required)
    )

# Pre-cache EfficientNetV2-s ImageNet weight so torch.hub/model_zoo does not hit
# the dead OneDrive URL during model initialization.
torch_cache_dir = Path.home() / '.cache' / 'torch' / 'hub' / 'checkpoints'
torch_cache_dir.mkdir(parents=True, exist_ok=True)
for name in external_pretrained_required:
    cached = torch_cache_dir / name
    src = CHECKPOINT_DIR / name
    if not cached.exists():
        try:
            os.symlink(src, cached)
        except OSError:
            shutil.copy2(src, cached)
        print(f'Cached external pretrained weight: {src} -> {cached}')

!ls -lh "$CHECKPOINT_DIR"


In [ ]:
# 4. Patch official code for modern Kaggle without changing the EVQA weights/architecture.
# - keep official EVQA/mPLUG feature stack
# - avoid loading full StableDiffusionPipeline; load only the same tokenizer/text_encoder
# - patch transformers>=4.57 import locations used by old mPLUG code
# - force torch.load(..., weights_only=False) for old checkpoints on new PyTorch

script_path = OFFICIAL_ECR_DIR / 'test_SnapUGC_baseline.py'
text = script_path.read_text()
if PATCH_LIGHT_SD_TEXT_ENCODER and 'StableDiffusionPipeline.from_pretrained' in text:
    old_sd = '''from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained(
        "CompVis/stable-diffusion-v1-4"
    )
model = EVQA(3, 16, pipe.tokenizer, pipe.text_encoder)'''
    new_sd = '''from transformers import CLIPTokenizer, CLIPTextModel
_sd_model_id = "CompVis/stable-diffusion-v1-4"
_sd_tokenizer = CLIPTokenizer.from_pretrained(_sd_model_id, subfolder="tokenizer")
_sd_text_encoder = CLIPTextModel.from_pretrained(_sd_model_id, subfolder="text_encoder").cuda().eval()
model = EVQA(3, 16, _sd_tokenizer, _sd_text_encoder)'''
    text = text.replace(old_sd, new_sd)
    print('Patched official script to avoid loading full StableDiffusionPipeline.')

# New PyTorch may default to weights_only=True. The official checkpoints are trusted here.
load_replacements = [
    ("torch.load(\"checkpoints/EVQA.pth\")", "torch.load(\"checkpoints/EVQA.pth\", weights_only=False)"),
    ("torch.load(\"checkpoints/mPLUG2_MSRVTT_Caption.pth\", map_location='cpu')", "torch.load(\"checkpoints/mPLUG2_MSRVTT_Caption.pth\", map_location='cpu', weights_only=False)"),
    ("torch.load(\"checkpoints/net_distort6_g_latest.pth\")", "torch.load(\"checkpoints/net_distort6_g_latest.pth\", weights_only=False)"),
    ("torch.load(\"checkpoints/r3d18_K_200ep.pth\")", "torch.load(\"checkpoints/r3d18_K_200ep.pth\", weights_only=False)"),
]
for old, new in load_replacements:
    text = text.replace(old, new)

# Current CLIPTextModel may not register position_ids as a loadable buffer.
# The official checkpoint can safely ignore this unexpected non-parameter key.
text = text.replace(
    "model.load_state_dict(torch.load(\"checkpoints/EVQA.pth\", weights_only=False)['params'])",
    "model.load_state_dict(torch.load(\"checkpoints/EVQA.pth\", weights_only=False)['params'], strict=False)",
)
text = text.replace(
    "model.load_state_dict(torch.load(\"checkpoints/EVQA.pth\")['params'])",
    "model.load_state_dict(torch.load(\"checkpoints/EVQA.pth\", weights_only=False)['params'], strict=False)",
)
script_path.write_text(text)

modeling_path = OFFICIAL_ECR_DIR / 'mPLUG_2' / 'models' / 'modeling_mplug2.py'
modeling_text = modeling_path.read_text()
old_import = '''from transformers.modeling_utils import (
    PreTrainedModel,
    apply_chunking_to_forward,
    find_pruneable_heads_and_indices,
    prune_linear_layer,
)'''
new_import = '''from transformers.modeling_utils import PreTrainedModel
try:
    from transformers.modeling_utils import (
        apply_chunking_to_forward,
        find_pruneable_heads_and_indices,
        prune_linear_layer,
    )
except ImportError:
    from transformers.pytorch_utils import (
        apply_chunking_to_forward,
        find_pruneable_heads_and_indices,
        prune_linear_layer,
    )'''
if old_import in modeling_text:
    modeling_text = modeling_text.replace(old_import, new_import)
    modeling_path.write_text(modeling_text)
    print('Patched mPLUG transformers import compatibility.')
else:
    print('mPLUG import block was already patched or changed upstream.')

# transformers>=4.57 calls get_vocab() during PreTrainedTokenizer.__init__.
# The old mPLUG tokenizer initialized self.vocab after super(), so move vocab
# loading before super() without changing tokenization behavior.
tokenizer_path = OFFICIAL_ECR_DIR / 'mPLUG_2' / 'models' / 'tokenization_bert.py'
tokenizer_text = tokenizer_path.read_text()
old_tokenizer_init = """        super().__init__(
            do_lower_case=do_lower_case,
            do_basic_tokenize=do_basic_tokenize,
            never_split=never_split,
            unk_token=unk_token,
            sep_token=sep_token,
            pad_token=pad_token,
            cls_token=cls_token,
            mask_token=mask_token,
            tokenize_chinese_chars=tokenize_chinese_chars,
            strip_accents=strip_accents,
            **kwargs,
        )

        if not os.path.isfile(vocab_file):
            raise ValueError(
                "Can't find a vocabulary file at path '{}'. To load the vocabulary from a Google pretrained "
                "model use `tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)`".format(vocab_file)
            )
        self.vocab = load_vocab(vocab_file)
        self.ids_to_tokens = collections.OrderedDict([(ids, tok) for tok, ids in self.vocab.items()])"""
new_tokenizer_init = """        if not os.path.isfile(vocab_file):
            raise ValueError(
                "Can't find a vocabulary file at path '{}'. To load the vocabulary from a Google pretrained "
                "model use `tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)`".format(vocab_file)
            )
        self.vocab = load_vocab(vocab_file)
        self.ids_to_tokens = collections.OrderedDict([(ids, tok) for tok, ids in self.vocab.items()])

        super().__init__(
            do_lower_case=do_lower_case,
            do_basic_tokenize=do_basic_tokenize,
            never_split=never_split,
            unk_token=unk_token,
            sep_token=sep_token,
            pad_token=pad_token,
            cls_token=cls_token,
            mask_token=mask_token,
            tokenize_chinese_chars=tokenize_chinese_chars,
            strip_accents=strip_accents,
            **kwargs,
        )"""
if old_tokenizer_init in tokenizer_text:
    tokenizer_path.write_text(tokenizer_text.replace(old_tokenizer_init, new_tokenizer_init))
    print('Patched mPLUG BertTokenizer vocab init compatibility.')
elif 'self.vocab = load_vocab(vocab_file)\n        self.ids_to_tokens' in tokenizer_text:
    print('mPLUG BertTokenizer vocab init patch was already applied or changed upstream.')
else:
    raise RuntimeError('Could not patch mPLUG tokenization_bert.py; inspect tokenizer init manually.')

!grep -n "CLIPTokenizer\|StableDiffusionPipeline\|model = EVQA\|weights_only\|pytorch_utils\|self.vocab" "$script_path" "$modeling_path" "$tokenizer_path" | head -60


In [ ]:
# 5. Prepare official CSV format: Id,Title,Description,Download_link
official_input_csv = RUN_DIR / 'official_input_5000.csv'
df = pd.read_csv(SUBSET_CSV)
for col in ['Id', 'Title', 'Description']:
    if col not in df.columns:
        raise ValueError(f'Missing required column: {col}')
if 'Download_link' not in df.columns:
    df['Download_link'] = ''
df[['Id', 'Title', 'Description', 'Download_link']].fillna('').to_csv(official_input_csv, index=False)
labels_csv = RUN_DIR / 'labels_5000.csv'
df[['Id', 'ECR']].to_csv(labels_csv, index=False)
print('official_input_csv:', official_input_csv)
print('labels_csv:', labels_csv)
print(pd.read_csv(official_input_csv).head())
print('video count:', len(list(SUBSET_VIDEO_DIR.glob('*.mp4'))))

In [ ]:
# 6. Run official SnapUGC EVQA inference.
# This is slow: it runs mPLUG-2, YAMNet, EfficientNetV2, Distortion, ResNet3D, and EVQA over 5000 videos.
import subprocess

submission_path = OFFICIAL_ECR_DIR / 'submission_baseline.csv'
if submission_path.exists():
    submission_path.unlink()  # remove stale sample submission bundled in official repo

cmd = [
    sys.executable,
    'test_SnapUGC_baseline.py',
    '--videos_dir', str(SUBSET_VIDEO_DIR),
    '--csv_file', str(official_input_csv),
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(OFFICIAL_ECR_DIR), env={**os.environ, 'PYTHONUNBUFFERED': '1'})
if result.returncode != 0:
    raise RuntimeError(f'Official SnapUGC inference failed with exit code {result.returncode}. Do not use any existing submission file.')
if not submission_path.exists():
    raise FileNotFoundError(submission_path)

out_submission = RUN_DIR / 'official_submission_baseline.csv'
shutil.copy2(submission_path, out_submission)
pred_df_check = pd.read_csv(out_submission)
input_ids = set(pd.read_csv(official_input_csv)['Id'].astype(str))
pred_ids = set(pred_df_check['Id'].astype(str))
missing_ids = sorted(input_ids - pred_ids)[:10]
extra_ids = sorted(pred_ids - input_ids)[:10]
print('Saved:', out_submission)
print('prediction rows:', len(pred_df_check), 'expected:', len(input_ids))
print(pred_df_check.head())
if len(pred_df_check) != len(input_ids) or missing_ids or extra_ids:
    raise RuntimeError(f'Prediction IDs do not match official_input_csv. missing={missing_ids}, extra={extra_ids}')


In [ ]:
# 7. Evaluate official prediction against the 5000 ECR labels
from scipy.stats import pearsonr, spearmanr, kendalltau

pred_df = pd.read_csv(RUN_DIR / 'official_submission_baseline.csv')
label_df = pd.read_csv(labels_csv)
pred_df['Id'] = pred_df['Id'].astype(str)
label_df['Id'] = label_df['Id'].astype(str)
merged = pred_df.merge(label_df, on='Id', suffixes=('_pred', '_true'))
if len(merged) != MAX_VIDEOS:
    raise RuntimeError(f'Expected {MAX_VIDEOS} evaluated rows, got {len(merged)}. Do not trust metrics.')
pred = merged['ECR_pred'].astype(float).to_numpy()
true = merged['ECR_true'].astype(float).to_numpy()
plcc = pearsonr(pred, true)[0] if pred.std() > 0 and true.std() > 0 else 0.0
srcc = spearmanr(pred, true).correlation
ktau = kendalltau(pred, true).correlation
plcc_clean = float(0.0 if np.isnan(plcc) else plcc)
srcc_clean = float(0.0 if np.isnan(srcc) else srcc)
metrics = {
    'n_eval': int(len(merged)),
    'plcc': plcc_clean,
    'srcc': srcc_clean,
    'ktau': float(0.0 if np.isnan(ktau) else ktau),
    'final_score_srcc06_plcc04': float(0.6 * srcc_clean + 0.4 * plcc_clean),
    'final_score_mean_plcc_srcc': float(0.5 * (plcc_clean + srcc_clean)),
    'final_score_formula': '0.6*SRCC + 0.4*PLCC',
    'mse': float(np.mean((pred - true) ** 2)),
    'mae': float(np.mean(np.abs(pred - true))),
    'pred_mean': float(pred.mean()),
    'pred_std': float(pred.std()),
    'true_mean': float(true.mean()),
    'true_std': float(true.std()),
}
metrics['final_score'] = metrics['final_score_srcc06_plcc04']
report = {
    'source': 'official dasongli1/SnapUGC_Engagement ECR_inference',
    'subset_csv': str(SUBSET_CSV),
    'subset_video_dir': str(SUBSET_VIDEO_DIR),
    'official_input_csv': str(official_input_csv),
    'submission': str(RUN_DIR / 'official_submission_baseline.csv'),
    'metrics': metrics,
}
report_path = RUN_DIR / 'official_evqa_report.json'
report_path.write_text(json.dumps(report, indent=2))
print(json.dumps(metrics, indent=2))
print('report_path:', report_path)


In [ ]:
# 8. Package only lightweight outputs. Do not zip videos/checkpoints.
import zipfile
zip_path = WORK_DIR / f'official_snapugc_evqa_{MAX_VIDEOS}_outputs.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in [SUBSET_CSV, RUN_DIR / 'official_input_5000.csv', RUN_DIR / 'labels_5000.csv', RUN_DIR / 'official_submission_baseline.csv', RUN_DIR / 'official_evqa_report.json']:
        if Path(path).exists():
            zf.write(path, arcname=str(Path(path).relative_to(OUTPUT_DIR)))
print('zip_path:', zip_path)
print('zip size MB:', round(zip_path.stat().st_size / (1024 * 1024), 2))